In [10]:
# ============================================================
# AI DOCUMENT ASSISTANT
# Google Colab - One Cell Application
# Required Colab Secret name: gen
# ============================================================

!pip install -q -U gradio pypdf google-genai reportlab

import os
import re
import hashlib
from datetime import datetime

import gradio as gr
from pypdf import PdfReader
from google import genai
from google.genai import types
from google.colab import userdata

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.colors import HexColor
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer


# ============================================================
# SETTINGS
# ============================================================

SECRET_NAME = "gen"

# Main Gemini model for this project
MODEL_NAME = "gemini-3.5-flash-lite"

# Fallback options tried automatically if a model is unavailable.
# The system moves to the next model only if the previous one returns 404.
MODEL_FALLBACKS = [
    "gemini-3.5-flash",
    "gemini-3.8-flash",
    "gemini-flash-latest",
]

MAX_CONTEXT_CHARS = 28000
MAX_ANALYSIS_CHARS = 45000
MAX_OUTPUT_TOKENS = 1600

try:
    GEMINI_API_KEY = userdata.get(SECRET_NAME)
except Exception:
    GEMINI_API_KEY = ""


# ============================================================
# APP STATE
# ============================================================

STATE = {
    "document_text": "",
    "document_name": "",
    "page_count": 0,
    "word_count": 0,
    "character_count": 0,
    "page_chunks": [],
    "summary": "",
    "key_points": "",
    "study_questions": "",
    "chat_history": [],
    "cache": {},
    "active_model": MODEL_NAME,
}


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def reset_state():
    STATE["document_text"] = ""
    STATE["document_name"] = ""
    STATE["page_count"] = 0
    STATE["word_count"] = 0
    STATE["character_count"] = 0
    STATE["page_chunks"] = []
    STATE["summary"] = ""
    STATE["key_points"] = ""
    STATE["study_questions"] = ""
    STATE["chat_history"] = []
    STATE["cache"] = {}
    STATE["active_model"] = MODEL_NAME


def normalize_text(text):
    return re.sub(r"\s+", " ", text or "").strip()


def get_client():
    if not GEMINI_API_KEY:
        return None

    return genai.Client(api_key=GEMINI_API_KEY)


def cache_key(task_name, prompt):
    raw = f"{task_name}|{STATE['document_name']}|{STATE['active_model']}|{prompt}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


def get_cached(task_name, prompt):
    return STATE["cache"].get(cache_key(task_name, prompt))


def save_cached(task_name, prompt, result):
    STATE["cache"][cache_key(task_name, prompt)] = result
    return result


def is_error_response(text):
    return text.startswith("## ❌") or text.startswith("## ⚠️")


# ============================================================
# ERROR MESSAGES
# ============================================================

def missing_api_key_message():
    return """
## ❌ Gemini API key not found

This app reads your Gemini API key from **Google Colab Secrets**.

Please check the following:

1. Click the **🔑 Secrets** icon in the left Colab sidebar.
2. Create a secret named exactly:

```text
gen
```

3. Paste your Gemini API key into its Value field.
4. Turn **Notebook access** ON.
5. Run this cell again.
"""


def quota_reached_message():
    return """
## ⚠️ Gemini API quota reached

Your Gemini project has reached its current API quota for this model.

This is not a PDF error and your application is working correctly.

### You can still use
- Document statistics
- PDF text preview
- Local PDF search
- Question history
- Downloading a report based on analyses already generated

### What to do
1. Wait for your Gemini quota to reset.
2. Check usage and limits in Google AI Studio.
3. Make fewer AI requests per day.
4. Enable billing if you need higher request limits.
"""


def model_unavailable_message():
    tried_models = [MODEL_NAME] + MODEL_FALLBACKS

    models_text = "\n".join(
        [f"- `{model}`" for model in tried_models]
    )

    return f"""
## ❌ No configured Gemini model is available

The application tried these model names:

{models_text}

Your API project may not have access to these models yet.

### What to do
1. Open Google AI Studio and check available models for your API key.
2. Check whether your API project is restricted by region, quota, or billing.
3. Replace `MODEL_NAME` near the top of the notebook with an available model.
"""


def generic_error_message(error):
    return f"""
## ❌ Gemini request failed

```text
{str(error)}
```
"""


# ============================================================
# GEMINI CALL WITH MODEL FALLBACK
# ============================================================

def call_gemini(task_name, prompt, output_tokens=MAX_OUTPUT_TOKENS):
    cached = get_cached(task_name, prompt)

    if cached:
        return cached

    client = get_client()

    if client is None:
        return missing_api_key_message()

    models_to_try = [STATE["active_model"]]

    for model in MODEL_FALLBACKS:
        if model not in models_to_try:
            models_to_try.append(model)

    last_error = None

    for model in models_to_try:
        try:
            response = client.models.generate_content(
                model=model,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.2,
                    max_output_tokens=output_tokens,
                ),
            )

            result = response.text or "Gemini did not return a response."

            STATE["active_model"] = model

            return save_cached(task_name, prompt, result)

        except Exception as error:
            last_error = error
            error_text = str(error)

            if "429" in error_text or "RESOURCE_EXHAUSTED" in error_text:
                return quota_reached_message()

            if "401" in error_text or "403" in error_text or "API_KEY" in error_text:
                return missing_api_key_message()

            if "404" in error_text or "not found" in error_text.lower():
                continue

            return generic_error_message(error)

    return model_unavailable_message() if last_error else generic_error_message("Unknown error")


# ============================================================
# PDF LOADING
# ============================================================

def load_pdf(file_path):
    reset_state()

    if not file_path:
        return (
            "⚠️ Please upload a PDF first.",
            "",
            "",
            "",
            "",
            "",
            "",
            gr.update(value=None),
            "",
        )

    try:
        reader = PdfReader(file_path)

        page_chunks = []
        document_parts = []

        for page_number, page in enumerate(reader.pages, start=1):
            page_text = (page.extract_text() or "").strip()

            page_chunks.append({
                "page_number": page_number,
                "text": page_text,
            })

            document_parts.append(
                f"\n--- PAGE {page_number} ---\n{page_text}"
            )

        full_text = "\n".join(document_parts).strip()
        document_name = os.path.basename(file_path)
        word_count = len(re.findall(r"\b[\w'-]+\b", full_text))

        STATE["document_text"] = full_text
        STATE["document_name"] = document_name
        STATE["page_count"] = len(reader.pages)
        STATE["word_count"] = word_count
        STATE["character_count"] = len(full_text)
        STATE["page_chunks"] = page_chunks

        if not normalize_text(full_text):
            return (
                f"""
## ⚠️ PDF loaded, but no readable text was found

**File:** `{document_name}`
**Pages:** {len(reader.pages)}

This appears to be a scanned or image-only PDF. The current version extracts selectable text, but scanned documents require OCR.
""",
                "",
                "",
                "",
                "",
                "",
                "",
                gr.update(value=None),
                "",
            )

        statistics = f"""
### 📊 Document Statistics

| Item | Value |
|---|---:|
| File | `{document_name}` |
| Pages | {len(reader.pages)} |
| Words extracted | {word_count:,} |
| Characters extracted | {len(full_text):,} |
| Preferred model | `{MODEL_NAME}` |
"""

        status = f"""
## ✅ Document loaded successfully

**{document_name}** is ready for analysis.

Use the tabs to generate a summary, identify key insights, ask questions, search the document, create study questions, or download a report.
"""

        preview = full_text[:3000]

        if len(full_text) > 3000:
            preview += "\n\n*Preview shortened. The full extracted document remains available for search and AI analysis.*"

        return (
            status,
            statistics,
            "Click **Generate Executive Summary** to analyze this PDF.",
            "Click **Extract Key Insights** to analyze this PDF.",
            "Ask a question in the **Ask PDF** tab.",
            "Search the PDF in the **Search** tab.",
            preview,
            gr.update(value=None),
            "",
        )

    except Exception as error:
        return (
            f"""
## ❌ Unable to read the PDF

```text
{error}
```

Please upload a valid, non-password-protected PDF file.
""",
            "",
            "",
            "",
            "",
            "",
            "",
            gr.update(value=None),
            "",
        )


# ============================================================
# RETRIEVAL: FIND RELEVANT PAGES FOR QUESTIONS
# ============================================================

def get_relevant_context(question, max_chars=MAX_CONTEXT_CHARS):
    if not STATE["page_chunks"]:
        return ""

    keywords = [
        word.lower()
        for word in re.findall(r"\b[\w'-]+\b", question)
        if len(word) >= 3
    ]

    scored_pages = []

    for item in STATE["page_chunks"]:
        page_number = item["page_number"]
        page_text = item["text"]
        lower_text = page_text.lower()

        score = sum(lower_text.count(keyword) for keyword in keywords)
        scored_pages.append((score, page_number, page_text))

    scored_pages.sort(key=lambda item: item[0], reverse=True)

    selected_pages = []
    total_characters = 0

    for _, page_number, page_text in scored_pages:
        page_block = f"\n--- PAGE {page_number} ---\n{page_text}\n"

        if total_characters + len(page_block) > max_chars:
            continue

        selected_pages.append(page_block)
        total_characters += len(page_block)

        if total_characters >= max_chars:
            break

    if not selected_pages:
        return STATE["document_text"][:max_chars]

    return "\n".join(selected_pages)


# ============================================================
# AI FEATURES
# ============================================================

def generate_summary():
    if not STATE["document_text"]:
        return "⚠️ Please upload and load a PDF first."

    if STATE["summary"]:
        return STATE["summary"]

    document_context = STATE["document_text"][:MAX_ANALYSIS_CHARS]

    prompt = f"""
You are an expert document analyst.

Create a polished, professional executive summary based only on the document text below.
Do not use external knowledge or invent facts.

Use this exact Markdown structure:

# Executive Summary
Write a concise 1–2 paragraph overview of the document.

## Main Topics
Give 5–8 clear bullet points.

## Key Findings
List the important conclusions, requirements, decisions, risks, facts, or outcomes.

## Important Details
Include useful dates, figures, names, organizations, and terms only if they appear in the document.

## Intended Audience
State who would gain the most value from this document.

Mention source page numbers when they are clearly available.

DOCUMENT NAME: {STATE["document_name"]}

DOCUMENT TEXT:
{document_context}
"""

    result = call_gemini("summary", prompt, output_tokens=1800)

    if not is_error_response(result):
        STATE["summary"] = result

    return result


def generate_key_points():
    if not STATE["document_text"]:
        return "⚠️ Please upload and load a PDF first."

    if STATE["key_points"]:
        return STATE["key_points"]

    document_context = STATE["document_text"][:MAX_ANALYSIS_CHARS]

    prompt = f"""
You are a professional research assistant.

Extract the most valuable structured information from the document below.
Use only the document text and never introduce outside facts.

Use this exact Markdown structure:

# Key Insights
Give 10 concise, non-repetitive bullet points.

## Important Terms and Concepts
Create a Markdown table with these columns:

| Term | Meaning in This Document |

## Dates, Numbers, and Named Entities
List important dates, numbers, people, organizations, places, or titles.
If there are none, write: Not clearly stated.

## Actions, Requirements, and Recommendations
List instructions, responsibilities, warnings, actions, recommendations, and next steps.
If there are none, write: Not clearly stated.

DOCUMENT NAME: {STATE["document_name"]}

DOCUMENT TEXT:
{document_context}
"""

    result = call_gemini("key_points", prompt, output_tokens=2000)

    if not is_error_response(result):
        STATE["key_points"] = result

    return result


def generate_study_questions():
    if not STATE["document_text"]:
        return "⚠️ Please upload and load a PDF first."

    if STATE["study_questions"]:
        return STATE["study_questions"]

    document_context = STATE["document_text"][:40000]

    prompt = f"""
You are a study coach.

Create a useful learning guide based only on the document below.

Use this exact Markdown structure:

# Suggested Questions
Write 12 questions that progress from basic understanding to critical analysis.

## Quick Revision Quiz
Write 5 short-answer questions.

## Discussion Prompts
Write 3 thoughtful discussion prompts.

Do not answer any of the questions.
Use only the document content.

DOCUMENT NAME: {STATE["document_name"]}

DOCUMENT TEXT:
{document_context}
"""

    result = call_gemini("study_questions", prompt, output_tokens=1600)

    if not is_error_response(result):
        STATE["study_questions"] = result

    return result


def ask_pdf(question):
    question = (question or "").strip()

    if not STATE["document_text"]:
        return "⚠️ Please upload and load a PDF first."

    if not question:
        return "⚠️ Please enter a question."

    context = get_relevant_context(question)

    prompt = f"""
You are a precise and professional PDF question-answering assistant.

Answer the user's question using only the supplied PDF excerpts.

Rules:
- Never use external knowledge.
- If the answer cannot be found, say exactly:
  "I could not find that information in the document."
- Keep the response clear, direct, and useful.
- Use bullets or a table when that makes the answer easier to understand.
- Cite page numbers only if they appear in the supplied excerpts.
- Never invent facts, quotations, or page numbers.

DOCUMENT NAME: {STATE["document_name"]}

PDF EXCERPTS:
{context}

USER QUESTION:
{question}
"""

    answer = call_gemini("question", prompt, output_tokens=1400)

    if not is_error_response(answer):
        STATE["chat_history"].append({
            "question": question,
            "answer": answer,
            "timestamp": datetime.now().strftime("%H:%M"),
        })

    return answer


# ============================================================
# LOCAL FEATURES — NO GEMINI QUOTA USED
# ============================================================

def search_pdf(search_term):
    search_term = (search_term or "").strip()

    if not STATE["document_text"]:
        return "⚠️ Please upload and load a PDF first."

    if not search_term:
        return "⚠️ Please enter a word or phrase to search."

    query = search_term.lower()
    matches = []

    for item in STATE["page_chunks"]:
        page_number = item["page_number"]
        page_text = item["text"]
        lower_text = page_text.lower()

        start_index = 0
        matches_on_page = 0

        while True:
            index = lower_text.find(query, start_index)

            if index == -1:
                break

            snippet_start = max(0, index - 180)
            snippet_end = min(len(page_text), index + len(search_term) + 280)

            snippet = page_text[snippet_start:snippet_end]
            snippet = snippet.replace("\n", " ").strip()

            matches.append(
                f"""
### Page {page_number}

…{snippet}…
"""
            )

            matches_on_page += 1
            start_index = index + len(query)

            if matches_on_page >= 3:
                break

    if not matches:
        return f"""
## 🔍 No results found

No exact match was found for:

```text
{search_term}
```
"""

    visible_matches = matches[:20]

    output = f"""
# 🔍 Search Results

**Search term:** `{search_term}`
**Showing:** {len(visible_matches)} of {len(matches)} result(s)

---

"""

    output += "\n---\n".join(visible_matches)

    if len(matches) > len(visible_matches):
        output += (
            f"\n\n*{len(matches) - len(visible_matches)} additional result(s) "
            "are not displayed.*"
        )

    return output


def get_question_history():
    if not STATE["chat_history"]:
        return "No questions have been asked yet."

    history = "# Question History\n\n"

    for index, item in enumerate(STATE["chat_history"], start=1):
        history += (
            f"## {index}. Question — {item['timestamp']}\n\n"
            f"**Question:** {item['question']}\n\n"
            f"**Answer:**\n\n{item['answer']}\n\n---\n\n"
        )

    return history


# ============================================================
# REPORT GENERATION
# ============================================================

def parse_markdown_for_report(markdown_text):
    parsed_items = []

    for line in markdown_text.splitlines():
        line = line.strip()

        if not line:
            parsed_items.append(("space", ""))
            continue

        safe_line = (
            line.replace("&", "&amp;")
            .replace("<", "&lt;")
            .replace(">", "&gt;")
        )

        if safe_line.startswith("# "):
            parsed_items.append(("heading", safe_line[2:].strip()))
        elif safe_line.startswith("## "):
            parsed_items.append(("subheading", safe_line[3:].strip()))
        elif safe_line.startswith("- "):
            parsed_items.append(("bullet", safe_line[2:].strip()))
        elif re.match(r"^\d+\.\s+", safe_line):
            parsed_items.append(("bullet", re.sub(r"^\d+\.\s+", "", safe_line)))
        elif safe_line.startswith("|"):
            continue
        else:
            safe_line = re.sub(r"\*\*(.*?)\*\*", r"<b>\1</b>", safe_line)
            parsed_items.append(("body", safe_line))

    return parsed_items


def create_report():
    if not STATE["document_text"]:
        return None, "⚠️ Please upload and load a PDF first."

    available_sections = []

    if STATE["summary"]:
        available_sections.append(("Executive Summary", STATE["summary"]))

    if STATE["key_points"]:
        available_sections.append(("Key Insights", STATE["key_points"]))

    if STATE["study_questions"]:
        available_sections.append(("Study Questions", STATE["study_questions"]))

    if not available_sections:
        return None, """
## ⚠️ No analysis is available for the report

Generate at least one of the following first:

- Executive Summary
- Key Insights
- Study Questions

The report uses results already generated, so it does not make another Gemini request.
"""

    report_path = "/content/AI_Document_Assistant_Report.pdf"

    document = SimpleDocTemplate(
        report_path,
        pagesize=A4,
        rightMargin=45,
        leftMargin=45,
        topMargin=50,
        bottomMargin=50,
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        "ReportTitle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontSize=20,
        leading=24,
        textColor=HexColor("#1F4E79"),
        spaceAfter=16,
    )

    heading_style = ParagraphStyle(
        "ReportHeading",
        parent=styles["Heading2"],
        fontSize=14,
        leading=18,
        textColor=HexColor("#1F4E79"),
        spaceBefore=12,
        spaceAfter=8,
    )

    body_style = ParagraphStyle(
        "ReportBody",
        parent=styles["BodyText"],
        fontSize=10.5,
        leading=15,
        spaceAfter=7,
    )

    bullet_style = ParagraphStyle(
        "ReportBullet",
        parent=body_style,
        leftIndent=14,
        firstLineIndent=-8,
    )

    story = []

    story.append(Paragraph("AI Document Assistant Report", title_style))

    metadata = (
        f"<b>Document:</b> {STATE['document_name']}<br/>"
        f"<b>Pages:</b> {STATE['page_count']}<br/>"
        f"<b>Words extracted:</b> {STATE['word_count']:,}<br/>"
        f"<b>Generated:</b> {datetime.now().strftime('%d %B %Y, %I:%M %p')}"
    )

    story.append(Paragraph(metadata, body_style))
    story.append(Spacer(1, 0.18 * inch))

    for section_name, section_content in available_sections:
        story.append(Paragraph(section_name, heading_style))

        for item_type, item_text in parse_markdown_for_report(section_content):
            if item_type == "space":
                story.append(Spacer(1, 0.05 * inch))
            elif item_type in {"heading", "subheading"}:
                story.append(Paragraph(item_text, heading_style))
            elif item_type == "bullet":
                story.append(Paragraph(f"• {item_text}", bullet_style))
            else:
                story.append(Paragraph(item_text, body_style))

        story.append(Spacer(1, 0.12 * inch))

    document.build(story)

    return report_path, "✅ Your professional analysis report is ready to download."


# ============================================================
# CLEAR APPLICATION
# ============================================================

def clear_application():
    reset_state()

    return (
        gr.update(value=None),
        "No PDF loaded yet.",
        "",
        "Click **Generate Executive Summary** after loading a PDF.",
        "Click **Extract Key Insights** after loading a PDF.",
        "Ask a question after loading a PDF.",
        "Search results will appear here.",
        "Load a PDF to see extracted text.",
        gr.update(value=None),
        "No report is available yet.",
        "",
        "",
        "No questions have been asked yet.",
    )


# ============================================================
# GRADIO 6 USER INTERFACE
# ============================================================

custom_css = """
.gradio-container {
    max-width: 1250px !important;
    margin: 0 auto !important;
}
#app-title {
    text-align: center;
}
#app-subtitle {
    text-align: center;
    color: #52606D;
    margin-bottom: 22px;
}
"""

with gr.Blocks(title="AI Document Assistant") as demo:

    gr.Markdown(
        """
        <div id="app-title">

        # 📄 AI Document Assistant

        </div>

        <div id="app-subtitle">

        Upload a PDF, generate professional summaries, discover key insights, search content, ask grounded questions, and create a downloadable report.

        </div>
        """
    )

    with gr.Row():

        with gr.Column(scale=1, min_width=300):

            pdf_file = gr.File(
                label="Upload PDF",
                file_types=[".pdf"],
                type="filepath",
            )

            load_button = gr.Button(
                "📥 Load Document",
                variant="primary",
                size="lg",
            )

            clear_button = gr.Button(
                "🗑️ Clear Document",
                variant="secondary",
            )

            status_output = gr.Markdown("No PDF loaded yet.")
            statistics_output = gr.Markdown()

            gr.Markdown(
                """
                ### Quick Guide

                1. Upload a text-based PDF.
                2. Click **Load Document**.
                3. Use the tabs to analyze, search, and ask questions.
                4. Create a report after generating analysis.

                Your Gemini key is securely read from the Colab Secret named `gen`.
                """
            )

        with gr.Column(scale=2, min_width=550):

            with gr.Tabs():

                with gr.Tab("📌 Summary"):
                    summary_button = gr.Button(
                        "✨ Generate Executive Summary",
                        variant="primary",
                    )

                    summary_output = gr.Markdown(
                        "Click **Generate Executive Summary** after loading a PDF."
                    )

                with gr.Tab("🔑 Key Insights"):
                    key_points_button = gr.Button(
                        "📋 Extract Key Insights",
                        variant="primary",
                    )

                    key_points_output = gr.Markdown(
                        "Click **Extract Key Insights** after loading a PDF."
                    )

                with gr.Tab("💬 Ask PDF"):
                    question_input = gr.Textbox(
                        label="Ask a question about the PDF",
                        placeholder="Example: What are the main conclusions and where are they discussed?",
                        lines=3,
                    )

                    ask_button = gr.Button(
                        "Ask Document",
                        variant="primary",
                    )

                    answer_output = gr.Markdown(
                        "Ask a question after loading a PDF."
                    )

                with gr.Tab("🔍 Search"):
                    search_input = gr.Textbox(
                        label="Search exact text inside the PDF",
                        placeholder="Example: budget, conclusion, recommendation",
                    )

                    search_button = gr.Button(
                        "Search Document",
                        variant="primary",
                    )

                    search_output = gr.Markdown(
                        "Search results will appear here."
                    )

                with gr.Tab("🎓 Study Helper"):
                    study_button = gr.Button(
                        "Generate Study Questions",
                        variant="primary",
                    )

                    study_output = gr.Markdown(
                        "Generate revision and discussion questions from your document."
                    )

                with gr.Tab("📖 Document Text"):
                    document_text_output = gr.Markdown(
                        "Load a PDF to see extracted text."
                    )

                with gr.Tab("🕘 History"):
                    history_output = gr.Markdown(
                        "No questions have been asked yet."
                    )

                with gr.Tab("📥 Report"):
                    gr.Markdown(
                        """
                        Create a downloadable report from analyses you have already generated.

                        The report does not make a new Gemini request, helping reduce API usage.
                        """
                    )

                    report_button = gr.Button(
                        "Create Downloadable Report",
                        variant="primary",
                    )

                    report_file_output = gr.File(
                        label="Download Your Report",
                    )

                    report_status_output = gr.Markdown(
                        "No report is available yet."
                    )

    # ========================================================
    # EVENTS — must stay inside the gr.Blocks context
    # ========================================================

    load_button.click(
        fn=load_pdf,
        inputs=pdf_file,
        outputs=[
            status_output,
            statistics_output,
            summary_output,
            key_points_output,
            answer_output,
            search_output,
            document_text_output,
            report_file_output,
            report_status_output,
        ],
    )

    summary_button.click(
        fn=generate_summary,
        inputs=[],
        outputs=summary_output,
    )

    key_points_button.click(
        fn=generate_key_points,
        inputs=[],
        outputs=key_points_output,
    )

    study_button.click(
        fn=generate_study_questions,
        inputs=[],
        outputs=study_output,
    )

    ask_button.click(
        fn=ask_pdf,
        inputs=question_input,
        outputs=answer_output,
    ).then(
        fn=get_question_history,
        inputs=[],
        outputs=history_output,
    )

    question_input.submit(
        fn=ask_pdf,
        inputs=question_input,
        outputs=answer_output,
    ).then(
        fn=get_question_history,
        inputs=[],
        outputs=history_output,
    )

    search_button.click(
        fn=search_pdf,
        inputs=search_input,
        outputs=search_output,
    )

    search_input.submit(
        fn=search_pdf,
        inputs=search_input,
        outputs=search_output,
    )

    report_button.click(
        fn=create_report,
        inputs=[],
        outputs=[report_file_output, report_status_output],
    )

    clear_button.click(
        fn=clear_application,
        inputs=[],
        outputs=[
            pdf_file,
            status_output,
            statistics_output,
            summary_output,
            key_points_output,
            answer_output,
            search_output,
            document_text_output,
            report_file_output,
            report_status_output,
            question_input,
            search_input,
            history_output,
        ],
    )


# ============================================================
# GRADIO 6 LAUNCH
# theme and css belong in .launch() for Gradio 6
# ============================================================

demo.launch(
    share=True,
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="indigo",
        neutral_hue="slate",
    ),
    css=custom_css,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b178adcdb9182f19e8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
